In [1]:
from datasets import load_dataset

/opt/miniconda3/envs/GSM8/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd

In [3]:
dataset = load_dataset("openai/gsm8k", "main")

In [4]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})


In [5]:
print(dataset["train"])  # Print the first example in the training set

Dataset({
    features: ['question', 'answer'],
    num_rows: 7473
})


In [6]:
train_ds_pd = dataset["train"].to_pandas()
test_ds_pd = dataset["test"].to_pandas()

In [7]:
whole_ds_pd = pd.concat([train_ds_pd, test_ds_pd], ignore_index=True)

In [8]:
print(len(whole_ds_pd))  # Display the first few rows of the combined dataset

8792


In [9]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

In [10]:
questions = whole_ds_pd["question"].tolist()
answers =  whole_ds_pd["answer"].tolist()

In [11]:
# Split data (80% train, 10% validation, 10% test)
train_questions, temp_questions, train_answers, temp_answers = train_test_split(
    questions, answers, test_size=0.2, random_state=42
)
val_questions, test_questions, val_answers, test_answers = train_test_split(
    temp_questions, temp_answers, test_size=0.5, random_state=42
)

In [12]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [13]:
def tokenize_data(questions, answers):
    inputs = tokenizer(questions, padding="max_length", truncation=True, return_tensors="pt", max_length=128)
    labels = tokenizer(answers, padding="max_length", truncation=True, return_tensors="pt", max_length=128)
    return inputs, labels

train_inputs, train_labels = tokenize_data(train_questions, train_answers)
val_inputs, val_labels = tokenize_data(val_questions, val_answers)

In [14]:
from torch.utils.data import Dataset, DataLoader

In [15]:
class MathDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels

    def __len__(self):
        return len(self.inputs["input_ids"])

    def __getitem__(self, idx):
        return {
            "input_ids": self.inputs["input_ids"][idx],
            "attention_mask": self.inputs["attention_mask"][idx],
            "labels": self.labels["input_ids"][idx],
        }

In [16]:
train_dataset = MathDataset(train_inputs, train_labels)
val_dataset = MathDataset(val_inputs, val_labels)

In [17]:
batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

In [18]:
from transformers import GPT2LMHeadModel
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.to("mps")  # Move model to GPU if available

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [19]:
import torch
from tqdm import tqdm

In [20]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
epochs = 3

for epoch in range(epochs):
    model.train()
    for batch in tqdm(train_loader):
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch["input_ids"].to("mps"),
            attention_mask=batch["attention_mask"].to("mps"),
            labels=batch["labels"].to("mps"),
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    val_loss = 0
    for batch in tqdm(val_loader):
        with torch.no_grad():
            outputs = model(
                input_ids=batch["input_ids"].to("mps"),
                attention_mask=batch["attention_mask"].to("mps"),
                labels=batch["labels"].to("mps"),
            )
        val_loss += outputs.loss.item()
    print(f"Epoch {epoch}, Validation Loss: {val_loss / len(val_loader)}")

100%|██████████| 110/110 [00:37<00:00,  2.90it/s]


Epoch 0, Validation Loss: 4.056478040868586


100%|██████████| 110/110 [00:37<00:00,  2.94it/s]


Epoch 1, Validation Loss: 4.006772149692882


100%|██████████| 110/110 [00:37<00:00,  2.94it/s]

Epoch 2, Validation Loss: 3.9912933349609374


In [21]:
def generate_answer(question):
    inputs = tokenizer(question, return_tensors="pt").to("mps")
    outputs = model.generate(**inputs, max_length=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

question = "Natalia sold clips to 80 friends in April and half as many in May. How many clips did she sell in total?"
print(generate_answer(question))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Natalia sold clips to 80 friends in April and half as many in May. How many clips did she sell in total? sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold sold
